# ESCI Label Audit
End-to-end notebook: data exploration → prompt development → full audit run → results analysis.

In [ ]:
import sys
sys.path.insert(0, '..')  # make label_audit/ importable

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

import pandas as pd
import config
from src.data import load_example_products, filter_audit_set

## 1. Data Exploration

In [ ]:
df = load_example_products(config.DATA_DIR)
print(df.shape)
df.dtypes

In [ ]:
# Null rates for product text fields
text_cols = ['product_title', 'product_description', 'product_bullet_point']
df[text_cols].isnull().mean().rename('null_rate')

In [ ]:
# Count of 'E'-labeled rows for the three target queries
audit_set = filter_audit_set(df, config.TARGET_QUERIES)
print(f"Audit set: {len(audit_set)} rows")
audit_set.groupby('query').size().rename('count')

In [ ]:
# Inspect a sample — read product text before touching the LLM
sample = audit_set.sample(5, random_state=42)[['query', 'product_title', 'product_bullet_point']]
pd.set_option('display.max_colwidth', 200)
sample

## 2. Prompt Development

Run the LLM against a small subset and inspect raw outputs before committing to a full run.

In [ ]:
from src.llm import LLMClient

client = LLMClient(model=config.MODEL_NAME, prompt_path=config.PROMPT_PATH)

# Print the prompt template so you can see and edit it
print(config.PROMPT_PATH.read_text())

In [ ]:
# Test on a handful of rows — iterate on prompts/audit.txt until this looks right
dev_set = audit_set.sample(min(10, len(audit_set)), random_state=0)

dev_results = []
for _, row in dev_set.iterrows():
    result = client.audit_pair(
        query=row['query'],
        product_title=row.get('product_title', ''),
        product_description=row.get('product_description', ''),
        product_bullet_point=row.get('product_bullet_point', ''),
    )
    dev_results.append({'query': row['query'], 'product_title': row['product_title'], **result})

pd.DataFrame(dev_results)

## 3. Full Audit Run

In [ ]:
from src.auditor import run_audit
from src.output import write_results, print_summary

results = run_audit(audit_set, client)
path = write_results(results, config.OUTPUT_DIR)
print(f"Saved to {path}")

## 4. Results Analysis

In [ ]:
print_summary(results)

In [ ]:
# Full results table
results

In [ ]:
# Mislabeled pairs with original query for context
mislabeled = results[~results['label_accurate']].merge(
    audit_set[['query_id', 'product_id', 'query', 'product_title']],
    on=['query_id', 'product_id'],
)
mislabeled[['query_id', 'product_id', 'query', 'product_title', 'reformulated_query']]